# Executing Parallel Workflows Using CrewAI Agents

## What you'll learn

By the end of this notebook you will be able to:

1. Tell the difference between **serial (sequential)** and **parallel** task execution in CrewAI
2. Mark a task `async_execution=True` so the crew doesn't block on it before moving to the next task
3. Reason about what "running in parallel" actually means inside CrewAI's default sequential process
4. Read output from every task individually, not just the crew's final result

> **Companion notebook:** [`crewai-websearch-serial.ipynb`](crewai-websearch-serial.ipynb) builds the same researcher → writer shape but wires `task_write`'s `context=[task_research]`, forcing the writer to wait for the researcher. This notebook removes that dependency and marks the researcher's task `async_execution=True` instead, so both tasks run at the same time.

## Background: What is CrewAI?

[CrewAI](https://github.com/joaomdmoura/crewAI) is a Python framework for orchestrating **role-playing, autonomous AI agents**. Instead of asking a single LLM to do everything, you assign each agent a specific role, equip it with relevant tools, and let the agents collaborate on tasks — much like a team of human specialists.

### Serial vs. parallel, in one line

| | Serial (companion notebook) | Parallel (this notebook) |
|---|---|---|
| Dependency | `task_write` gets `context=[task_research]` | Neither task references the other |
| Execution | Writer waits for the researcher to finish | Researcher (`async_execution=True`) and writer run concurrently |
| Use when | One task's output must feed the next | Tasks are independent and can run at the same time |

---

## Scenario

A product leader at a global FMCG company requires timely insights on how AI is reshaping product development, consumer engagement, and supply chain efficiency. Manually gathering and summarizing this information is time-consuming and slows product strategy decisions.

We'll build two **independent** tasks — one researching AI-driven drug-discovery companies, another summarizing AI in patient diagnostics — and run them **at the same time** instead of one after another, since neither needs the other's output.

### The pipeline

```
Crew.kickoff_async()
        │
        ├──────────────────────────┬───────────────────────────┐
        ▼                          ▼                           │
┌─────────────────────┐   ┌──────────────────────┐              │
│ task_parallel_1      │   │ task_parallel_2       │             │
│ Researcher + Web     │   │ Writer                │             │
│ Search               │   │ (no context= link)    │             │
│ async_execution=True │   │ starts immediately    │             │
└──────────┬───────────┘   └──────────┬────────────┘             │
           │    both run concurrently  │                        │
           ▼                           ▼                        │
    companies list             diagnostics report                │
           │                           │                         │
           └─────────────┬─────────────┘                         │
                          ▼                                      │
               Crew waits for both to finish ◀────────────────────┘
                          │
                          ▼
                  parallel_result
```

Because neither task lists the other in `context=`, CrewAI doesn't need to serialize them — it kicks off `task_parallel_1` in the background (`async_execution=True`) and immediately starts `task_parallel_2` rather than waiting. Only once *every* task has finished does the crew assemble the final result.

## Step 1 — Install dependencies

Same libraries as the serial notebook:

| Package | Purpose |
|---|---|
| `crewai` | The main framework: Agent, Task, Crew, and Tool abstractions |
| `langchain`, `langchain-openai`, `langchain-community` | LangChain core + OpenAI & community integrations used internally by CrewAI |
| `langchain-tavily`, `tavily-python` | Bindings for the [Tavily](https://tavily.com) web-search API |
| `pydantic` | Powers CrewAI's `BaseTool` and typed models |
| `python-dotenv` | Loads secrets from a `.env` file so API keys never appear in the notebook |
| `litellm` | A unified interface to 100+ LLM providers; CrewAI uses it under the hood |

> **Before running:** create a `.env` file in this directory containing:
> ```
> OPENAI_API_KEY=sk-...
> OPENAI_MODEL_NAME=gpt-5-mini
> TAVILY_API_KEY=tvly-...
> ```
> You can get a free Tavily key at [app.tavily.com](https://app.tavily.com).

In [ ]:
!pip install crewai 
!pip install langchain langchain-openai langchain-community 
!pip install langchain-tavily tavily-python pydantic 
!pip install dotenv
!pip install litellm

## Step 2 — Import dependencies

- **`os`, `requests`** — read environment variables and make the raw HTTP call to Tavily
- **`dotenv.load_dotenv`** — parse the `.env` file and inject its values into `os.environ`
- **`crewai.Agent`, `Task`, `Crew`** — the three core building blocks
- **`crewai.llm.LLM`** — CrewAI's LLM wrapper (backed by LiteLLM)
- **`crewai.tools.BaseTool`** — the base class we subclass to create a custom tool

In [15]:
from dotenv import load_dotenv
import requests, os
from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai.tools import BaseTool

load_dotenv()  # Load environment variables from .env file

True

## Step 3 — Build the web-search tool and configure the LLM

Same `TavilySearchTool` pattern as the serial notebook: subclass `BaseTool`, set `name`/`description`, and implement `_run(query)` to call the Tavily API and return clean title/URL pairs.

Both agents share a single `LLM` instance below (`temperature=0.3`, favoring more consistent, less creative output). That's fine for this notebook — it's illustrating task *scheduling*, not per-agent token accounting, so there's no need for the separate `researcher_llm` / `writer_llm` instances used in the serial notebook.

In [16]:
class TavilySearchTool(BaseTool):
    name: str = "web-search"
    description: str = "Search the web for recent information"

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }
        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title}: {link}")

        return "\n".join(results) if results else "No results found."
    
search_tool = TavilySearchTool()

llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0.3,
)


## Step 4 — Define the agents

Same researcher/writer split as the serial notebook:

- **`researcher`** — has `tools=[search_tool]`, so it can search the web for drug-discovery companies
- **`writer`** — no tools; `allow_delegation=True` lets it hand off sub-work to another agent if it decides it needs to (not exercised in this scenario, but available)

What's different in this notebook isn't the agents — it's how their tasks are wired together next.

In [17]:
# Agents
researcher = Agent(
    role = "AI Researcher",
    goal = "Find the latest advancements n AI for healthcare",
    backstory = "You are an expert in AI and stay updated with the latest research and trends in healtcare.",
    verbose=True,
    llm=llm,
    tools=[search_tool]
)

writer = Agent(
    role = "Technical writer",
    goal = "Summarize reaserch into an excutive report, no more than 200 words:", 
    backstory = "You are an experienced technical writer with expertise in summarizing healthcare research for executives.",
    verbose=True,
    allow_delegation=True,
    llm=llm
)

## Step 5 — Wire up parallel tasks and run the crew

### `async_execution=True`

`task_parallel_1` (the researcher's company list) sets `async_execution=True`. Instead of blocking the crew until it finishes, CrewAI kicks it off in the background and immediately moves on to the next task in the list.

### Why `task_parallel_2` doesn't wait

`task_parallel_2` (the writer's diagnostics report) has no `context=` entry pointing at `task_parallel_1` — there's nothing for it to wait on, so it starts executing right away, in parallel with the researcher's task.

### What to look for in the output

- Both agents' verbose logs will interleave in the output, since they're genuinely running at the same time
- The crew still waits for **both** tasks to finish before returning — parallelism here means "don't block unnecessarily," not "return early"

In [ ]:
task_parallel_1 = Task(
    name="Researcher - Drug Discovery Companies",
    description="Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line about what they specialize in.",
    expected_output="Company names and what they specialize in.",
    agent=researcher,
    async_execution=True
)

task_parallel_2 = Task(
    name="Writer - Diagnostics Report",
    description="Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 200 words.",
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = await crew_parallel.kickoff_async()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)

## Step 6 — Inspect every task's own output

`parallel_result.raw` only shows **one** thing: the final task in the `tasks=[...]` list (here, the writer's diagnostics report). Since `task_parallel_1` and `task_parallel_2` ran independently, that single `.raw` string silently drops the researcher's parallel result entirely.

`CrewOutput.tasks_output` is the fix — a list of every task's own `TaskOutput`, in task order, regardless of which one actually finished first. Use it whenever you need to see parallel results side by side.

In [ ]:
print("All task outputs")
print("=" * 30)
for task_output in parallel_result.tasks_output:
    print(f"\n--- {task_output.name} ---")
    print(task_output.raw)